# Monday Morning at NorthStar

> *Sarah Chen · Customer Experience Analyst · NorthStar Retail · January 2023.*

**Estimated time: 10–15 minutes. Run every cell top to bottom.**

This is your first hands-on notebook. Run it **before class** — the goal is for you to *feel* what ML can do first, and *understand why* later.

## The Scene

It is 9:12 AM on a Monday in January 2023. You are **Sarah Chen**, starting your second week as Customer Experience Analyst at NorthStar Retail — a mid-sized online clothing store.

**Aisha Patel** from Customer Service walks into your cube holding a USB drive:

> *"Priya wants sentiment on every customer review we got in December. All ten thousand of them. By Friday. Apparently you're the new analyst so — good luck."*

She drops the drive on your desk and leaves.

You open the CSV. 10,000 rows. Reading one review takes about 15 seconds. That's **42 hours of reading** — more than a work-week — before you write a single line of analysis.

You could try a shortcut: write `if-else` rules that look for positive and negative words. Or you could try this "Machine Learning" thing you've been hearing about.

Let's try both.

## Step 1 — Set up

We'll use a lightweight sentiment model called **TextBlob**. It is a small library (no large model downloads, no internet required after install) that can classify a piece of English text as positive or negative with one line of code.

In [ ]:
# Run this cell first. It installs TextBlob if needed, then loads it.
# Nothing to edit — just click this cell and press Shift+Enter.
try:
    from textblob import TextBlob
except ImportError:
    print("TextBlob not found — installing now (one-time, ~10 seconds)...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "textblob"])
    from textblob import TextBlob
    print("✅ Installed.")

import pandas as pd
print("✅ Libraries ready.")

## Step 2 — The reviews

Here are 5 real-looking reviews from NorthStar's December haul. Read them carefully before running the next cell — try to guess each one's sentiment in your head.

In [ ]:
reviews = [
    "Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again.",
    "The shirt arrived two weeks late and the color is nothing like the photo. Very disappointed.",
    "Great value for the price. Not amazing but not bad — I'd order again for basics.",
    "I ordered size M as usual and it's enormous. Runs at least two sizes large. Returning.",
    "Sure, it's 'fine', if you enjoy paying $85 for something that comes apart after one wash.",
]

for i, r in enumerate(reviews, 1):
    print(f'{i}. {r}\n')

### Pause & Predict

Before running the next cell, predict in your head:
- Which of the 5 reviews is **positive**?
- Which is **negative**?
- Is any one of them **tricky**?

Write your predictions down (even just mentally) before scrolling.

## Step 3 — Attempt 1: hand-written rules

Let's try the traditional approach. We'll write a rule: if a review contains more positive words than negative words, label it positive.

This is how programmers solved problems before ML. Watch what happens.

In [ ]:
POSITIVE_WORDS = ['love', 'great', 'amazing', 'good', 'perfect', 'premium']
NEGATIVE_WORDS = ['bad', 'disappointed', 'late', 'enormous', 'terrible', 'worst']

def rule_based_sentiment(text):
    text_lower = text.lower()
    pos = sum(1 for w in POSITIVE_WORDS if w in text_lower)
    neg = sum(1 for w in NEGATIVE_WORDS if w in text_lower)
    if pos > neg:
        return 'positive'
    elif neg > pos:
        return 'negative'
    else:
        return 'neutral / unclear'

for i, r in enumerate(reviews, 1):
    label = rule_based_sentiment(r)
    print(f'{i}. [{label:17}]  {r[:70]}...')

### What do you notice?

Look carefully at what the rule-based classifier labelled each review. Compare to your own predictions.

- Review 3 ("Great value... not amazing but not bad") — does the rule handle mixed opinions well?
- Review 5 ("Sure, it's 'fine', if you enjoy paying $85...") — this is sarcasm. Did the rule catch it?
- What other words would you need to add to the lists? When would this ever end?

**This is the core problem the rule-based approach was never going to solve.** You cannot write a finite rulebook for language. People are too creative.

## Step 4 — Attempt 2: Machine Learning

Now let's try ML. TextBlob ships with a pre-trained sentiment model — someone else trained it on millions of labelled English sentences. We use it with one line.

**No training. No rulebook. One line.**

In [ ]:
def ml_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity  # a score from -1 (negative) to +1 (positive)
    if polarity > 0.1:
        return 'positive', polarity
    elif polarity < -0.1:
        return 'negative', polarity
    else:
        return 'neutral', polarity

print(f"{'#':3} {'label':10} {'score':>6}  review")
print('-' * 90)
for i, r in enumerate(reviews, 1):
    label, score = ml_sentiment(r)
    print(f'{i:<3} {label:10} {score:>6.2f}  {r[:60]}...')

### What do you notice?

Compare the ML output to your own predictions, and to the rule-based classifier.

- The score (called **polarity**) is between -1 and +1, not just a label. It tells you *how* positive or negative, not just *which*.
- **Look closely at review 5** (the sarcastic one: *"Sure, it's 'fine', if you enjoy paying $85..."*). What did the model label it? Is that right?
- **Look at review 4** (the sizing complaint). The word 'enormous' isn't negative by itself — the model may have been unsure.
- Notice that you did not have to teach the model about *every* word. It generalised from the training data it saw when it was originally trained.

**Key insight:** The ML model is better than the rules *on average*, but it **still makes mistakes** — especially on sarcasm, negation, and context-heavy text. It is a big improvement over rules, not a miracle. Keep that in mind for the next step.

## Step 5 — The punch line

Now let's run the ML model on **all 10,000 reviews**, not just 5. We'll use a simulated CSV that represents Sarah's actual workload.

In [ ]:
# Simulate 10k reviews by repeating and varying our 5 seeds
import random
random.seed(42)

variants = [
    '',
    ' Arrived in good condition.',
    ' Shipping was fine.',
    ' Would recommend to a friend.',
    ' Not what I expected.',
    ' Thanks for the fast delivery!',
    ' The packaging could be better.',
]

big_batch = [random.choice(reviews) + random.choice(variants) for _ in range(10_000)]
print(f'Total reviews to classify: {len(big_batch):,}')

In [ ]:
import time

start = time.time()
labels = [ml_sentiment(r)[0] for r in big_batch]
elapsed = time.time() - start

print(f'Classified {len(big_batch):,} reviews in {elapsed:.1f} seconds.')
print()

result = pd.Series(labels).value_counts()
print('Breakdown:')
print(result.to_string())

### What do you notice?

- **Manual effort for this task:** ~42 hours of reading.
- **ML effort for this task:** what you just saw — a few seconds.
- Sarah's Friday deadline just became trivial. The rest of the week she can spend on *what this report means*, not on reading.

This is why ML matters in business: **scale that humans cannot match, on problems where a human expert would do it if only they had the time.**

## Step 6 — Sarah's moment of doubt

Sarah writes up her findings and walks into Priya's office with the numbers she just saw.

> *Sarah: "Roughly 60% of December reviews are positive, 20% negative, 20% neutral."*

Priya reads the numbers. Then looks up:

> *Priya: "Sarah — are the positive ones **actually** positive? And how do we know the model isn't miscounting? You mentioned earlier it got a sarcastic review wrong. How many more of those are in the 10,000?"*

Sarah opens her mouth and closes it again. She doesn't know.

She can see the model ran. She can see the numbers. But she has no way to *prove* the numbers are right. The sarcastic review from Step 4 is just one error she caught by hand. How many silent errors are hiding in the other 9,995 reviews?

That question — *how sure are we?* — is the single most important question in applied ML. It is the question Lesson 2 will teach you to answer. And Lessons 3 and 4. And every lesson after that, in some form.

**For now, bring Priya's question to class.** The instructor will help you start to answer it.

## ✅ End of Notebook

You have just completed the first step of your pre-class work.

**Next step →** Go back to [`pre-class.md`](../pre-class.md) and continue with Step 2 (Reflect). The whole pre-class cycle takes ~75 minutes. You've used about 15.

When you get to class, Part 1 will revisit what you just saw — but with a real, heavyweight model (Hugging Face transformers) and a harder set of reviews.